# Chapter 6 — Domain Applications: Chemistry & Beyond

**Learning objectives**
- Understand second quantization and why it maps to qubits naturally
- Apply the Jordan-Wigner transform to convert fermionic operators to Pauli strings
- Set up a minimal molecular Hamiltonian (H₂) manually
- Run VQE on H₂ and track energy convergence toward the exact ground state
- Use `qsharp.estimate()` for a chemistry problem (from the df-chemistry sample)
- Survey `Std.TableLookup` and fixed-point arithmetic for scientific computing
- Understand block encoding of Hamiltonians

## Setup

In [ ]:
import qsharp
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
print(f"qsharp {__import__("importlib.metadata", fromlist=["version"]).version("qsharp")}")

## 6.1 Second Quantization and Fermionic Operators

In second quantization, a quantum chemistry Hamiltonian is written as:

$$H = \sum_{pq} h_{pq}\, a_p^\dagger a_q + \frac{1}{2}\sum_{pqrs} h_{pqrs}\, a_p^\dagger a_q^\dagger a_r a_s$$

where $a_p^\dagger$ (create) and $a_p$ (annihilate) are fermionic operators satisfying the anti-commutation relations:

$$\{a_p, a_q^\dagger\} = \delta_{pq}, \quad \{a_p, a_q\} = 0$$

The one-electron integrals $h_{pq}$ encode kinetic energy + electron-nuclear attraction; the two-electron integrals $h_{pqrs}$ encode electron-electron repulsion.

### H₂ minimal basis (STO-3G)

With 2 spatial orbitals × 2 spins = 4 spin-orbitals. The Hamiltonian has 15 Pauli terms after Jordan-Wigner mapping (at a specific bond length R = 0.74 Å):

In [ ]:
# H₂ Hamiltonian in STO-3G basis at R=0.74 Å (pre-computed)
# Coefficients from: Whitfield et al. (2011), Molecular Physics 109(5):735-750
H2_terms = [
    # (pauli_string, coefficient)
    ("IIII", -0.8126179630197575),
    ("IIIZ", +0.1711977490486789),
    ("IIZI", -0.2227979949377701),
    ("IZII", +0.1711977490486789),
    ("ZIII", -0.2227979949377701),
    ("IIZZ", +0.1686280558804425),
    ("IZIZ", +0.1205448220109419),
    ("IZZI", +0.1659278503699195),
    ("ZIIZ", +0.1659278503699195),
    ("ZIZI", +0.1205448220109419),
    ("ZZII", +0.1686280558804425),
    ("XXYY", -0.04533108862490958),
    ("XYYX", +0.04533108862490958),
    ("YXXY", +0.04533108862490958),
    ("YYXX", -0.04533108862490958),
]

print("H₂ Pauli Hamiltonian (15 terms):")
for pauli, coef in H2_terms:
    print(f"  {coef:+.6f} * {pauli}")

print(f"\nFCI ground state energy (exact): -1.1372838 Hartree")

## 6.2 Jordan-Wigner Transform

The Jordan-Wigner (JW) transform maps fermionic operators to qubit operators:

$$a_p^\dagger = \left(\prod_{j<p} Z_j\right) \frac{X_p - iY_p}{2}, \quad a_p = \left(\prod_{j<p} Z_j\right) \frac{X_p + iY_p}{2}$$

This ensures that anti-commutation relations are preserved via the Z-string (Jordan-Wigner string). One qubit per spin-orbital.

In [ ]:
def jw_creator(p, n_modes):
    """Jordan-Wigner creation operator a†_p as a sum of Pauli strings."""
    # Returns list of (coeff, pauli_string)
    z_string = ['Z'] * p + ['I'] * (n_modes - p)

    x_term = z_string[:]
    x_term[p] = 'X'

    y_term = z_string[:]
    y_term[p] = 'Y'

    return [
        (0.5, ''.join(x_term)),        # 0.5 * Z...ZXI...I
        (-0.5j, ''.join(y_term)),      # -0.5i * Z...ZYI...I
    ]

n = 4  # H₂ has 4 spin-orbitals
print("JW creation operator a†_1 (spin-orbital 1):")
for coef, pauli in jw_creator(1, n):
    print(f"  {coef:.2f} * {pauli}")

print("\nJW creation operator a†_2 (spin-orbital 2):")
for coef, pauli in jw_creator(2, n):
    print(f"  {coef:.2f} * {pauli}")

## 6.3 VQE on H₂

We implement VQE using the UCCSD ansatz. For H₂ in STO-3G, a single-parameter ansatz suffices:

$$|\psi(\theta)\rangle = e^{\theta(a_0^\dagger a_1^\dagger a_2 a_3 - h.c.)} |\text{HF}\rangle$$

The Hartree-Fock reference state for H₂ is |1100⟩ (both electrons in the lowest two spin-orbitals).

In [ ]:
%%qsharp

import Std.Arrays.*;
import Std.Convert.*;
import Std.Math.*;

/// Hartree-Fock state for H₂: |1100⟩ (2 electrons in 4 spin-orbitals)
operation PrepareHF(qs : Qubit[]) : Unit is Adj {
    X(qs[0]); X(qs[1]);  // Occupy spin-orbitals 0 and 1
}

/// UCCSD singles+doubles excitation for H₂ (1-parameter)
/// Excites electrons from orbitals 0,1 → 2,3
operation UCC_Excitation(theta : Double, qs : Qubit[]) : Unit is Adj {
    // Implement e^{theta * (XXYY + XYYX + YXXY + YYXX) / 8}
    // This is the double-excitation gate for (0,1) → (2,3)
    let angle = theta / 8.0;
    // XXYY term
    within {
        H(qs[0]); H(qs[1]);
        Adjoint S(qs[2]); H(qs[2]);
        Adjoint S(qs[3]); H(qs[3]);
    } apply {
        CNOT(qs[0], qs[1]); CNOT(qs[1], qs[2]); CNOT(qs[2], qs[3]);
        Rz(angle, qs[3]);
        CNOT(qs[2], qs[3]); CNOT(qs[1], qs[2]); CNOT(qs[0], qs[1]);
    }
}

/// Ansatz: |ψ(θ)⟩ = UCC_Excitation(θ) |HF⟩
operation H2Ansatz(theta : Double, qs : Qubit[]) : Unit is Adj {
    PrepareHF(qs);
    UCC_Excitation(theta, qs);
}

/// Measure one Pauli term ⟨ψ|P|ψ⟩ given a basis string and coefficient
/// Basis: array of Pauli for each qubit
operation MeasurePauliTerm(theta : Double, basis : Pauli[], shots : Int) : Double {
    mutable zeroCount = 0;
    for _ in 1..shots {
        use qs = Qubit[4];
        H2Ansatz(theta, qs);
        if Measure(basis, qs) == Zero { zeroCount += 1; }
        ResetAll(qs);
    }
    // Expectation: +1 for Zero, -1 for One → 2*pZero - 1
    2.0 * IntAsDouble(zeroCount) / IntAsDouble(shots) - 1.0
}

In [ ]:
def pauli_to_qsharp(pauli_str):
    """Convert 'XYIZ' string to Q# Pauli array literal."""
    mapping = {'I': 'PauliI', 'X': 'PauliX', 'Y': 'PauliY', 'Z': 'PauliZ'}
    return '[' + ', '.join(mapping[c] for c in pauli_str) + ']'

def compute_h2_energy(theta, shots=300):
    """Compute H₂ VQE energy at ansatz parameter theta."""
    energy = 0.0
    for pauli_str, coef in H2_terms:
        if all(p == 'I' for p in pauli_str):
            energy += coef  # Identity term contributes directly
        else:
            basis = pauli_to_qsharp(pauli_str)
            exp_val = qsharp.eval(f"MeasurePauliTerm({theta:.6f}, {basis}, {shots})")
            energy += coef * exp_val
    return energy

# Scan theta to find the minimum
print("Scanning ansatz parameter θ...")
thetas = np.linspace(-np.pi, np.pi, 15)
energies = [compute_h2_energy(t, shots=80) for t in thetas]

min_idx = np.argmin(energies)
print(f"Minimum energy: {energies[min_idx]:.6f} Hartree at θ = {thetas[min_idx]:.4f} rad")
print(f"Exact FCI:      -1.137284 Hartree")

plt.figure(figsize=(9, 4))
plt.plot(thetas, energies, 'o-', markersize=4, label='VQE energy')
plt.axhline(-1.1372838, color='red', linestyle='--', label='FCI ground state')
plt.xlabel("θ (radians)")
plt.ylabel("Energy (Hartree)")
plt.title("VQE energy surface for H₂ (STO-3G)")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

## 6.4 Block Encoding of Hamiltonians

Block encoding is a technique that embeds a Hamiltonian H (subnormalized) into the top-left block of a unitary:

$$U = \begin{pmatrix} H/\lambda & * \\ * & * \end{pmatrix}$$

where λ is the 1-norm of H. This enables quantum signal processing, qubitization, and fault-tolerant phase estimation. The QDK chemistry library provides a complete Jordan-Wigner block encoding.

Reference: `library/chemistry/src/JordanWigner/BlockEncoding.qs`

In [ ]:
# Conceptual sketch — block encoding via LCU (Linear Combination of Unitaries)
print("Block encoding structure for H₂ Hamiltonian:")
print()
print("H = Σᵢ αᵢ Uᵢ  (Linear Combination of Unitaries)")
print()

# Compute 1-norm (normalization factor λ)
lam = sum(abs(coef) for _, coef in H2_terms)
print(f"Number of Pauli terms: {len(H2_terms)}")
print(f"1-norm λ = Σ|αᵢ| = {lam:.4f} Hartree")
print()
print("The block encoding uses:")
print(f"  - {int(np.ceil(np.log2(len(H2_terms))))} ancilla qubits to select among {len(H2_terms)} terms (PREP circuit)")
print(f"  - 4 system qubits for H₂")
print(f"  - SELECT circuit applies the selected Pauli string")
print(f"  - PREP† ∘ SELECT ∘ PREP implements the LCU walk operator")

## 6.5 Resource Estimation for Chemistry

For fault-tolerant quantum chemistry, we estimate resources for the double-factorized (DF) block encoding approach, which significantly reduces T-gate count compared to naive approaches.

Reference: `samples/estimation/df-chemistry/`

In [ ]:
# Set up for resource estimation
qsharp.init(target_profile=qsharp.TargetProfile.Base)

# A simple proxy for chemistry resource estimation:
# QPE of a block-encoded Hamiltonian requires:
# - T-count ≈ (lam / epsilon) * T_per_PREP
# where epsilon is target energy accuracy

# For H₂ with λ=2.84 and targeting 1 mHartree accuracy:
epsilon = 1e-3  # Hartree (chemical accuracy)
T_per_PREP = 100  # rough estimate

print("Resource scaling for quantum chemistry:")
print()
molecules = [
    ("H₂",      4,   2.84,   100),
    ("H₂O",    14,  62.0,    400),
    ("N₂",     28, 150.0,    800),
    ("Benzene", 72, 820.0,  3000),
    ("FeMo-co",114,1600.0,  8000),
]
print(f"{'Molecule':>10} {'Spin-orbs':>10} {'λ (Ha)':>10} {'T/PREP':>8} {'T total (M)':>12}")
print("-" * 55)
for name, n_orbs, lam_mol, t_prep in molecules:
    n_steps = int(np.pi * lam_mol / (2 * epsilon))
    t_total = n_steps * t_prep / 1e6
    print(f"{name:>10} {n_orbs:>10} {lam_mol:>10.1f} {t_prep:>8} {t_total:>12.1f}")

print()
print("See samples/estimation/df-chemistry/ for full DF resource estimation notebooks.")

In [ ]:
%%qsharp

// Simple stand-in for a chemistry oracle: LCU walk operator for 4-qubit system
// (mimics structure without importing the full chemistry library)
operation ChemistryOracle(qs : Qubit[]) : Unit is Adj + Ctl {
    // Approximate single Trotter step for H₂ (IZ, ZI, ZZ, XX, YY terms)
    let dt = 0.01;
    Rz(0.1711 * dt, qs[0]);
    Rz(-0.2228 * dt, qs[1]);
    Rz(0.1711 * dt, qs[2]);
    Rz(-0.2228 * dt, qs[3]);
    Rzz(0.1686 * dt, qs[0], qs[2]);
    Rzz(0.1659 * dt, qs[0], qs[3]);
    Rxx(0.0453 * dt, qs[0], qs[2]);
    Ryy(0.0453 * dt, qs[0], qs[2]);
}

operation RunChemistryTrotter(nSteps : Int) : Unit {
    use qs = Qubit[4];
    // Start from HF state |1100⟩
    X(qs[0]); X(qs[1]);
    for _ in 1..nSteps {
        ChemistryOracle(qs);
    }
    ResetAll(qs);
}

In [ ]:
result = qsharp.estimate("RunChemistryTrotter(10)")
print("Resource estimate for 10-step Trotter simulation of H₂ proxy:")
print(f"  Physical qubits: {result['physicalCounts']['physicalQubits']:,}")
print(f"  T-gate count:    {result['logicalCounts']['tCount']:,}")
print(f"  Runtime:         {result['physicalCounts']['runtime'] * 1e-9:.3f} seconds")

## 6.6 Fixed-Point Arithmetic and Table Lookups

Scientific computing on quantum hardware requires arithmetic that works within quantum constraints. The QDK provides:

- **Fixed-point arithmetic** (`Std.FixedPoint`): quantum registers representing rational numbers, supporting addition, multiplication, and comparisons
- **Table lookups** (`Std.TableLookup`): efficiently load classical data into quantum registers using QROM (quantum read-only memory)

In [ ]:
%%qsharp

import Std.TableLookup.Select;
import Std.Arrays.IndexRange;
import Std.Diagnostics.DumpMachine;
import Std.Convert.IntAsDouble;

/// Table lookup demo: quantum-controlled classical table read
/// Implements |i⟩|0⟩ → |i⟩|table[i]⟩
operation TableLookupDemo() : Unit {
    // Classical lookup table: molecular energy at 5 bond lengths (in mHartree)
    let table = [[false, false, true],   // entry 0: 001 = 1
                 [false, true, false],   // entry 1: 010 = 2
                 [false, true, true],    // entry 2: 011 = 3
                 [true, false, false]];  // entry 3: 100 = 4

    use address = Qubit[2];   // 2-bit address for 4 entries
    use data = Qubit[3];      // 3-bit data register

    // Put address in superposition over all 4 entries
    H(address[0]); H(address[1]);

    Message("Address register in superposition:");

    // QROM lookup: |address⟩|0⟩ → |address⟩|table[address]⟩
    Select(table, address, data);

    Message("After QROM lookup (address entangled with data):");
    DumpMachine();

    ResetAll(address + data);
}

TableLookupDemo()

In [ ]:
# Switch back to Unrestricted for Ising model section (uses mid-circuit measurement + classical control)
qsharp.init(target_profile=qsharp.TargetProfile.Unrestricted)

## 6.7 Extending the Library for New Domains

The QDK chemistry library (`library/chemistry/`) follows a layered architecture:

```
library/chemistry/src/JordanWigner/
    Data.qs             ← JWEncodingData, JWInputState structs
    EvolutionSet.qs     ← Trotter steps for JW terms
    BlockEncoding.qs    ← LCU block encoding
    StatePreparation.qs ← Prepare trial states
    VQE.qs              ← EstimateEnergy, MeasurementOperators
```

To extend for a new domain (e.g., condensed matter physics, materials science):
1. Define your Hamiltonian in a `Data.qs` equivalent with your qubit encoding
2. Implement a `EvolutionSet` that applies each Hamiltonian term
3. Write a `BlockEncoding` using `Select` (QROM) for PREP + `EvolutionSet` for SELECT
4. Connect to `EstimateEnergy` for VQE or to `ApplyQPE` for fault-tolerant QPE

In [ ]:
%%qsharp
// Ising model Hamiltonian on a 1D chain: H = -J * sum ZZ - h * sum X

import Std.Arrays.IndexRange;
import Std.Convert.IntAsDouble;

/// One Trotter step for the transverse-field Ising model
/// H = -J * Σ ZᵢZᵢ₊₁ - h * Σ Xᵢ
operation IsingTrotterStep(
    qs : Qubit[],
    J : Double,
    h : Double,
    dt : Double
) : Unit is Adj + Ctl {
    let n = Length(qs);
    // ZZ interactions
    for i in 0..n-2 {
        Rzz(-2.0 * J * dt, qs[i], qs[i+1]);
    }
    // Transverse field (X terms)
    for q in qs {
        Rx(-2.0 * h * dt, q);
    }
}

operation IsingGroundStateVQE(nQubits : Int, J : Double, h : Double, theta : Double) : Double {
    // Ansatz: parameterized Trotter layers
    mutable energy = 0.0;
    let shots = 60;

    // ZZ energy terms
    for i in 0..nQubits-2 {
        mutable zzEnergy = 0.0;
        for _ in 1..shots {
            use qs = Qubit[nQubits];
            // Simple variational ansatz
            ApplyToEach(H, qs);
            IsingTrotterStep(qs, J, h, theta);
            let r = Measure([PauliZ, PauliZ], [qs[i], qs[i+1]]);
            if r == Zero { zzEnergy += 1.0; } else { zzEnergy -= 1.0; }
            ResetAll(qs);
        }
        energy -= J * zzEnergy / IntAsDouble(shots);
    }
    energy
}

In [ ]:
# Scan Trotter parameter for 4-qubit Ising model (J=1, h=0.5)
thetas = np.linspace(0.01, 1.5, 10)
ising_energies = [qsharp.eval(f"IsingGroundStateVQE(4, 1.0, 0.5, {t:.4f})") for t in thetas]

plt.figure(figsize=(8, 4))
plt.plot(thetas, ising_energies, 'o-')
plt.xlabel("Trotter step θ")
plt.ylabel("Variational energy")
plt.title("VQE for 4-qubit transverse-field Ising model (J=1, h=0.5)")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()
print(f"Minimum energy found: {min(ising_energies):.4f}")

## Summary

- **Second quantization**: fermionic creation/annihilation operators capture electron correlations; H = Σ hpq a†p aq + ...
- **Jordan-Wigner**: maps fermionic operators to Pauli strings via Z-strings; one qubit per spin-orbital
- **VQE on H₂**: single-parameter UCCSD ansatz reaches near-FCI accuracy on 4 qubits
- **Block encoding**: embeds H/λ in the top-left block of a unitary; enables qubitization and fault-tolerant QPE
- **Resource estimation**: T-count for chemistry scales with λ (1-norm) and target accuracy ε
- **Table lookups** (`Std.TableLookup.Select`): quantum-controlled classical data read — key for QROM in block encoding
- **Domain extension**: implement `Data`, `EvolutionSet`, `BlockEncoding` for new physics Hamiltonians